# Credential-free quickstart

This tutorial validates, inspects, and executes a version 1 program with a deterministic fake tool provider. It requires no model credentials or network access.

In [ ]:
from treelang import AST
from treelang.testing import FakeToolProvider, ToolCall

A serialized program is safe to store or review before execution. `AST.parse` applies the version 1 schema contract.

In [ ]:
program_json = {
    "type": "program",
    "body": [
        {
            "type": "function",
            "name": "double",
            "params": [{"type": "value", "name": "value", "value": 21}],
        }
    ],
    "mode": "single",
    "name": "Double 21",
    "description": "Return 42 using one deterministic tool call.",
    "schema_version": "1.0",
}
program = AST.parse(program_json)
print(AST.repr(program))

`FakeToolProvider` supplies metadata and local results while recording every call. Real applications inject their own `ToolProvider` implementation.

In [ ]:
provider = FakeToolProvider(
    [
        {
            "name": "double",
            "description": "Double a number.",
            "properties": {"value": {"type": "number"}},
        }
    ],
    results={"double": lambda arguments: arguments["value"] * 2},
)
result = await AST.eval(program, provider)
assert result == 42
assert provider.calls == [ToolCall("double", {"value": 21})]
result